# Document Reader Smoke Test: SQuAD Context -> Answer Span / No Answer

This notebook runs the small smoke test first. Retrieval is skipped for now: the reader receives each SQuAD 2.0 question together with the SQuAD `context` field as the passage.

Pipeline: `SQuAD question + SQuAD context -> Document Reader -> answer span or no answer`

In [ ]:
# Colab setup. Runtime > Change runtime type > GPU is recommended.
!pip -q install transformers datasets evaluate accelerate

In [ ]:
import collections
import os
import time

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, load_dataset
from torch import nn
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    BertModel,
    BertPreTrainedModel,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
from transformers.modeling_outputs import QuestionAnsweringModelOutput

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())

## Configuration

Keep this deliberately small. The goal is to verify preprocessing, training, post-processing, SQuAD v2 scoring, and the custom model forward pass before running larger experiments.

In [ ]:
MODEL_NAME = "bert-base-uncased"
SEED = 42
MAX_LENGTH = 384
DOC_STRIDE = 128
SMOKE_TRAIN_EXAMPLES = 100
SMOKE_VALIDATION_EXAMPLES = 100
OUTPUT_DIR = "./reader_smoke_outputs"

TRAINING_HYPERPARAMS = dict(
    learning_rate=3e-5,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load SQuAD 2.0 and Add a Replaceable Passage Provider

For now, `passage == context`. Later, only `add_reader_passages` needs to change when retrieval starts producing passages.

In [ ]:
raw_squad = load_dataset("rajpurkar/squad_v2")

train_smoke = raw_squad["train"].shuffle(seed=SEED).select(range(SMOKE_TRAIN_EXAMPLES))
validation_smoke = raw_squad["validation"].shuffle(seed=SEED).select(range(SMOKE_VALIDATION_EXAMPLES))
smoke_data = DatasetDict({"train": train_smoke, "validation": validation_smoke})


def add_reader_passages(batch):
    return {"passage": batch["context"]}

smoke_data = smoke_data.map(add_reader_passages, batched=True)
smoke_data

## Tokenization and Labels

This is the standard long-context QA setup: `max_length=384`, `stride=128`, overflow windows, and `[CLS]` labels for no-answer examples or answer spans outside a given window.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def prepare_train_features(examples):
    tokenized = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["passage"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    special_tokens_mask = tokenized.pop("special_tokens_mask")

    start_positions, end_positions, example_ids = [], [], []
    question_token_mask, context_token_mask = [], []

    for feature_index, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][feature_index]
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        answers = examples["answers"][sample_index]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        example_ids.append(examples["id"][sample_index])

        question_token_mask.append([
            int(sequence_ids[i] == 0 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(input_ids))
        ])
        context_token_mask.append([
            int(sequence_ids[i] == 1 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(input_ids))
        ])

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= answer_start and offsets[token_end_index][1] >= answer_end):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= answer_start:
                token_start_index += 1
            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1
            start_positions.append(token_start_index - 1)
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized["example_id"] = example_ids
    tokenized["question_token_mask"] = question_token_mask
    tokenized["context_token_mask"] = context_token_mask
    return tokenized


def prepare_validation_features(examples):
    tokenized = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["passage"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    special_tokens_mask = tokenized.pop("special_tokens_mask")
    example_ids, question_token_mask, context_token_mask = [], [], []

    for feature_index in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        example_ids.append(examples["id"][sample_index])
        question_token_mask.append([
            int(sequence_ids[i] == 0 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(tokenized["input_ids"][feature_index]))
        ])
        context_token_mask.append([
            int(sequence_ids[i] == 1 and special_tokens_mask[feature_index][i] == 0)
            for i in range(len(tokenized["input_ids"][feature_index]))
        ])
        tokenized["offset_mapping"][feature_index] = [
            offset if sequence_ids[i] == 1 else None
            for i, offset in enumerate(tokenized["offset_mapping"][feature_index])
        ]

    tokenized["example_id"] = example_ids
    tokenized["question_token_mask"] = question_token_mask
    tokenized["context_token_mask"] = context_token_mask
    return tokenized

train_features = smoke_data["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=smoke_data["train"].column_names,
)
validation_features = smoke_data["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=smoke_data["validation"].column_names,
)

print(train_features)
print(validation_features)

## Custom BERT + DrQA-Inspired Attention Reader

This model separates question and context tokens, aligns each context token to a soft attention-weighted question representation, combines context/question/interaction features, projects back to BERT hidden size, and predicts start/end logits. `[CLS]` remains the no-answer position.

In [ ]:
class BertDrQAQuestionAttentionForQA(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.similarity = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.projection = nn.Sequential(
            nn.Linear(config.hidden_size * 3, config.hidden_size),
            nn.GELU(),
            nn.LayerNorm(config.hidden_size),
            nn.Dropout(config.hidden_dropout_prob),
        )
        self.qa_outputs = nn.Linear(config.hidden_size, 2)
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        start_positions=None,
        end_positions=None,
        question_token_mask=None,
        context_token_mask=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]

        if question_token_mask is None:
            question_token_mask = ((token_type_ids == 0) & (attention_mask == 1)).long()
        if context_token_mask is None:
            context_token_mask = ((token_type_ids == 1) & (attention_mask == 1)).long()

        q_mask = question_token_mask.bool()
        c_mask = context_token_mask.bool()

        similarity_scores = torch.matmul(self.similarity(sequence_output), sequence_output.transpose(1, 2))
        similarity_scores = similarity_scores.masked_fill(~q_mask[:, None, :], torch.finfo(similarity_scores.dtype).min)
        attention_weights = torch.softmax(similarity_scores, dim=-1)
        aligned_question = torch.matmul(attention_weights, sequence_output)

        combined = torch.cat([sequence_output, aligned_question, sequence_output * aligned_question], dim=-1)
        enhanced_output = self.projection(combined)

        valid_answer_positions = c_mask.clone()
        valid_answer_positions[:, 0] = True
        enhanced_output = torch.where(valid_answer_positions[:, :, None], enhanced_output, sequence_output)

        logits = self.qa_outputs(enhanced_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1).contiguous()
        end_logits = end_logits.squeeze(-1).contiguous()

        invalid_positions = ~valid_answer_positions
        start_logits = start_logits.masked_fill(invalid_positions, torch.finfo(start_logits.dtype).min)
        end_logits = end_logits.masked_fill(invalid_positions, torch.finfo(end_logits.dtype).min)

        total_loss = None
        if start_positions is not None and end_positions is not None:
            ignored_index = start_logits.size(1)
            start_positions = start_positions.clamp(0, ignored_index)
            end_positions = end_positions.clamp(0, ignored_index)
            loss_fct = nn.CrossEntropyLoss(ignore_index=ignored_index)
            total_loss = (loss_fct(start_logits, start_positions) + loss_fct(end_logits, end_positions)) / 2

        if not return_dict:
            output = (start_logits, end_logits) + outputs[2:]
            return ((total_loss,) + output) if total_loss is not None else output

        return QuestionAnsweringModelOutput(
            loss=total_loss,
            start_logits=start_logits,
            end_logits=end_logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

## Post-Processing and Metrics

This converts token logits back to text spans and uses the official SQuAD v2 metric. It also splits results into answerable and unanswerable subsets.

In [ ]:
squad_v2_metric = evaluate.load("squad_v2")


def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = collections.OrderedDict()
    for example_index, example in enumerate(examples):
        min_null_score = None
        valid_answers = []
        context = example["passage"]

        for feature_index in features_per_example[example_index]:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            cls_score = start_logits[0] + end_logits[0]
            min_null_score = cls_score if min_null_score is None else min(min_null_score, cls_score)

            start_indexes = np.argsort(start_logits)[-1:-n_best_size - 1:-1].tolist()
            end_indexes = np.argsort(end_logits)[-1:-n_best_size - 1:-1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue
                    if offset_mapping[start_index] is None or offset_mapping[end_index] is None:
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue
                    start_char, _ = offset_mapping[start_index]
                    _, end_char = offset_mapping[end_index]
                    valid_answers.append({
                        "score": start_logits[start_index] + end_logits[end_index],
                        "text": context[start_char:end_char],
                    })

        best_answer = max(valid_answers, key=lambda x: x["score"]) if valid_answers else {"text": "", "score": 0.0}
        predictions[example["id"]] = "" if min_null_score is not None and min_null_score > best_answer["score"] else best_answer["text"]

    formatted_predictions = [
        {"id": example_id, "prediction_text": text, "no_answer_probability": float(text == "")}
        for example_id, text in predictions.items()
    ]
    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return formatted_predictions, references


def compute_squad_v2_metrics(predictions, references):
    overall = squad_v2_metric.compute(predictions=predictions, references=references)
    refs_by_id = {ref["id"]: ref for ref in references}
    ans_preds, ans_refs, no_preds, no_refs = [], [], [], []
    for pred in predictions:
        ref = refs_by_id[pred["id"]]
        if len(ref["answers"]["text"]) == 0:
            no_preds.append(pred); no_refs.append(ref)
        else:
            ans_preds.append(pred); ans_refs.append(ref)
    answerable = squad_v2_metric.compute(predictions=ans_preds, references=ans_refs) if ans_preds else {}
    unanswerable = squad_v2_metric.compute(predictions=no_preds, references=no_refs) if no_preds else {}
    return {
        "overall_em": overall.get("exact", 0.0),
        "overall_f1": overall.get("f1", 0.0),
        "answerable_em": answerable.get("exact", 0.0),
        "answerable_f1": answerable.get("f1", 0.0),
        "unanswerable_em": unanswerable.get("exact", 0.0),
        "unanswerable_f1": unanswerable.get("f1", 0.0),
    }


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Run Both Smoke Tests

Both models use identical data and hyperparameters. This should be enough to catch shape bugs, preprocessing mistakes, and metric/post-processing issues.

In [ ]:
def build_model(model_kind):
    if model_kind == "bert_baseline":
        return AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
    if model_kind == "bert_drqa_attention":
        return BertDrQAQuestionAttentionForQA.from_pretrained(MODEL_NAME)
    raise ValueError(f"Unknown model kind: {model_kind}")


def run_reader_smoke_test(model_kind):
    print(f"\n=== Running {model_kind} ===")
    model = build_model(model_kind)
    params = count_trainable_parameters(model)

    train_dataset = train_features.remove_columns(["example_id"])
    eval_dataset = validation_features.remove_columns(["example_id", "offset_mapping"])

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_kind),
        seed=SEED,
        data_seed=SEED,
        **TRAINING_HYPERPARAMS,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=default_data_collator,
    )

    start = time.perf_counter()
    trainer.train()
    train_time_sec = time.perf_counter() - start

    start = time.perf_counter()
    raw_predictions = trainer.predict(eval_dataset).predictions
    inference_time_sec = time.perf_counter() - start

    predictions, references = postprocess_qa_predictions(smoke_data["validation"], validation_features, raw_predictions)
    metrics = compute_squad_v2_metrics(predictions, references)

    result = {
        "model": model_kind,
        "train_examples": SMOKE_TRAIN_EXAMPLES,
        "validation_examples": SMOKE_VALIDATION_EXAMPLES,
        "parameters": params,
        "train_time_sec": round(train_time_sec, 2),
        "inference_time_sec": round(inference_time_sec, 2),
        **{k: round(v, 2) for k, v in metrics.items()},
    }
    print(result)
    return result

smoke_results = [run_reader_smoke_test(kind) for kind in ["bert_baseline", "bert_drqa_attention"]]
pd.DataFrame(smoke_results)

## After This Passes

Use the same functions for nested 10%, 30%, and 50% subsets. The reader models do not need to change when the passage provider switches from SQuAD context to retrieved text.